# Nemotron v1 SFT — Colab Pro+ A100

**Open in Colab Pro+, pick A100 GPU + High-RAM, then Run All.**

Repo: https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge

Nemotron v1 SFT — Colab Pro+ A100 version.

> USAGE (Colab Pro+):
> 1. New Notebook → Runtime → Change runtime type → A100 (40 GB GPU,
>    High-RAM). Save & connect.
> 2. Paste this whole script into one cell.
> 3. Run All. Expected runtime: 6-10 h on A100 40 GB.
> 4. The trained LoRA adapter lands at
>    /content/drive/MyDrive/nemotron-2026/adapter_v1/. After it
>    finishes, download that folder (or upload it as a Kaggle Dataset
>    `ky7240/nemotron-v1-adapter`) and reference it from a thin
>    submission notebook (= GPU off, just writes submission.csv with
>    the adapter path) to submit to Kaggle.

ATTRIBUTION (= Apache 2.0 licensed public kernels we draw setup code
from; please credit when publishing):
- dgxchen/training-with-unsloth-to-achieve-0-85-lb (Kaggle kernel)
- konbu17/nemotron-sft-lora-with-cot (Kaggle kernel)
- huikang/end-to-end-finetuning-for-lb-0-85 (Kaggle kernel)

NOVEL CONTRIBUTION = the SFT data (data/processed/train_sft_verifier_
only.jsonl, 5418 records). It is generated by 5 deterministic Python
solvers in this repo (Roman / Physics / Unit / Cipher / Bit) producing
verifier-backed Chain-of-Thought traces. See
docs/strategy/winning-strategy.dense.md for the full rationale.

## Cell 1 — Drive mount + GitHub clone

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge.git"
REPO_DIR = "/content/nemotron"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"])

ADAPTER_DIR = "/content/drive/MyDrive/nemotron-2026/adapter_v1"
os.makedirs(ADAPTER_DIR, exist_ok=True)

## Cell 2 — Dependencies (Unsloth official install pattern for Colab)

In [ ]:
# Use the Unsloth-recommended Colab install pattern. The single
# `pip install unsloth==xxx` form caused dependency conflicts (= torch,
# transformers) on Python 3.12. The split below installs Unsloth first
# (it pulls compatible torch/triton), then forces the latest nightly
# from GitHub with --no-deps so we don't downgrade Colab's CUDA torch.
# Reference: https://docs.unsloth.ai/get-started/installing-+-updating
import subprocess
import sys


def _run(cmd):
    """Run shell command, stream output to the cell so the user sees errors."""
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


# Step 1: install Unsloth + the heavy deps it knows about.
_run([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

# Step 2: force the nightly version + companion libs without resolving
# torch (Colab already ships a compatible CUDA torch wheel).
_run([sys.executable, "-m", "pip", "install", "-q",
      "--upgrade", "--no-deps", "--force-reinstall",
      "unsloth_zoo"])

# Step 3: trl + extras we depend on. transformers and peft come along
# with unsloth's pin so don't re-pin them here.
_run([sys.executable, "-m", "pip", "install", "-q",
      "trl", "datasets", "accelerate", "bitsandbytes",
      "sentencepiece", "nbformat"])

import torch

print(
    "cuda:",
    torch.cuda.is_available(),
    "device:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "mem GB:",
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0,
)

## Cell 3 — Load base model from HuggingFace + LoRA setup

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096  # = Kaggle inference max_model_len (rel. official metric)
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    attn_implementation="eager",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=MAX_SEQ_LEN,
)
print("✓ LoRA model ready (rank 32, max_seq_len", MAX_SEQ_LEN, ")")

## Cell 4 — Load SFT data + chat template + upsample + completion-only

In [ ]:
import json
from collections import Counter

from datasets import Dataset, concatenate_datasets

SFT_PATH = os.path.join(REPO_DIR, "data/processed/train_sft_verifier_only.jsonl")
assert os.path.exists(SFT_PATH), f"SFT data missing at {SFT_PATH}"
rows = [json.loads(l) for l in open(SFT_PATH)]
print(f"loaded {len(rows)} verifier-backed records")
print("source distribution:", Counter(r["source"] for r in rows))

INFERENCE_USER_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
)


def render(rec):
    user_content = rec["user"] + INFERENCE_USER_SUFFIX
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    return {"text": prompt_text + rec["assistant"]}


ds = Dataset.from_list([render(r) for r in rows])

# Codex review 3 §3 — upsample under-represented bit (× 13) and cipher
# (× 2) so they survive epoch-1 gradient mass.
upsample = {"solver_bit": 13, "solver_cipher": 2}
extras = []
for r in rows:
    n = upsample.get(r["source"], 1)
    if n > 1:
        extras.extend([r] * (n - 1))
if extras:
    extras_ds = Dataset.from_list([render(r) for r in extras])
    ds = concatenate_datasets([ds, extras_ds]).shuffle(seed=42)
print(f"after upsampling: {len(ds)} records")

## Cell 5 — SFT training (1 epoch, completion-only loss)

In [ ]:
from transformers import TrainingArguments
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer

args = TrainingArguments(
    output_dir="/content/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    warmup_ratio=0.03,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_steps=2000,
    save_total_limit=1,
    optim="adamw_8bit",
    weight_decay=0.0,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

# Completion-only loss masking: the response template is the last 40
# chars of the rendered prompt (= boundary tokens after which the
# assistant's rationale begins).
sample_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "X"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
response_template = sample_prompt[-40:]
print(f"completion-only response_template: {response_template!r}")
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=args,
    packing=False,
)
trainer.train()
print("✓ training done")

## Cell 6 — Save adapter to Drive

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter to", ADAPTER_DIR)
for fn in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fn))
    print(f"  {fn}: {sz:,} bytes")
print()
print("Next: upload this folder to Kaggle as `ky7240/nemotron-v1-adapter`,")
print("then push the thin submission notebook (= GPU off) to Kaggle and submit.")